In [1]:
import pandas as pd
import sys
import os

sys.path.append('../')
from src.Sql_pipeline_dataloader import load_cement_data

In [2]:
df = load_cement_data()

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
#statistical Model and ML models

##using SARIMAX

In [5]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [7]:
df.head()


,date,site_id,region,behavior,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,North,aggressive,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,North,aggressive,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,North,aggressive,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,North,aggressive,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,North,aggressive,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


In [8]:
site_id = 'SITE_001'
site_df = df[df['site_id'] == site_id].copy() # Filter the DataFrame for the specific site

In [9]:
df.head


<bound method NDFrame.head of             date   site_id region    behavior cement_type  \
0     2022-01-01  SITE_001  North  aggressive      CEM_II   
1     2022-01-02  SITE_001  North  aggressive       CEM_I   
2     2022-01-03  SITE_001  North  aggressive     CEM_III   
3     2022-01-04  SITE_001  North  aggressive       CEM_I   
4     2022-01-05  SITE_001  North  aggressive     CEM_III   
...          ...       ...    ...         ...         ...   
32875 2024-12-27  SITE_030  South  aggressive     CEM_III   
32876 2024-12-28  SITE_030  South  aggressive       CEM_I   
32877 2024-12-29  SITE_030  South  aggressive     CEM_III   
32878 2024-12-30  SITE_030  South  aggressive      CEM_II   
32879 2024-12-31  SITE_030  South  aggressive       CEM_I   

       planned_pour_tonnes  consumed_tonnes  opening_inventory_tonnes  \
0                    43.18            34.54                     52.56   
1                    45.26            45.26                     63.85   
2                 

In [10]:
site_df.set_index('date', inplace=True)
site_df = site_df.sort_index()  # Ensure the DataFrame is sorted by date

In [11]:
site_df.shape

(1096, 12)

In [12]:
site_df.columns

Index(['site_id', 'region', 'behavior', 'cement_type', 'planned_pour_tonnes',
       'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes',
       'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity'],
      dtype='str')

In [13]:
y = site_df['consumed_tonnes']  # Target variable

exog = site_df[['planned_pour_tonnes', 'rain_mm', 'avg_temp_c']]  # Exogenous variables for SARIMAX (with the 3 regressors)

In [14]:
split_index = int(len(site_df) * 0.8)

In [15]:
X_train, X_test = exog.iloc[:split_index], exog.iloc[split_index:] # Split the exogenous variables into training and testing sets
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]   # Split the target variable into training and testing sets

In [16]:
model = SARIMAX(     # Create the SARIMAX model
    y_train, exog=X_train, 
    order=(1, 1, 1), 
    seasonal_order=(1, 0, 1, 7),  # Assuming weekly seasonality)
    enforce_stationarity=False, 
    enforce_invertibility=False 
)   

c:\Users\obinw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\obinw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [17]:
results = model.fit(disp=False)  # Fit the SARIMAX model to the training data
results.summary()  # Display the summary of the fitted model

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                   consumed_tonnes   No. Observations:                  876
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3381.155
Date:                           Wed, 18 Nov 2026   AIC                           6778.310
Time:                                   11:08:26   BIC                           6816.421
Sample:                               01-01-2022   HQIC                          6792.895
                                    - 05-25-2024                                         
Covariance Type:                             opg                                         
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
planned_pour_tonnes     0.5504      0.039     14.239      0.000       0.475       0.626
rain_mm                -0.8606      0.067    -12.835      0.000      -0.992      -0.729
avg_temp_c              0.0497      0.052      0.962      0.336      -0.052       0.151
ar.L1                   0.0849      0.034      2.473      0.013       0.018       0.152
ma.L1                  -0.9970      0.004   -225.112      0.000      -1.006      -0.988
ar.S.L7                -0.0430      0.592     -0.073      0.942      -1.202       1.116
ma.S.L7                 0.0188      0.591      0.032      0.975      -1.140       1.177
sigma2                143.4096      8.812     16.275      0.000     126.139     160.681
===================================================================================
Ljung-Box (L1) (Q):                   0.00   Jarque-Bera (JB):                35.97
Prob(Q):                              1.00   Prob(JB):                         0.00
Heteroskedasticity (H):               0.93   Skew:                            -0.43
Prob(H) (two-sided):                  0.55   Kurtosis:                         2.48
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [18]:
print(results.summary())

                                     SARIMAX Results                                     
Dep. Variable:                   consumed_tonnes   No. Observations:                  876
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3381.155
Date:                           Wed, 18 Nov 2026   AIC                           6778.310
Time:                                   11:08:26   BIC                           6816.421
Sample:                               01-01-2022   HQIC                          6792.895
                                    - 05-25-2024                                         
Covariance Type:                             opg                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
planned_pour_tonnes     0.5504      0.039     14.239      0.000       0.475       0.626
rain_mm         

In [19]:
sarimax_forecast = results.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)  # Generate forecasts for the test period i.e making predictions for the test set using the fitted model and the exogenous variables from the test set

In [20]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import plotly.graph_objects as go

fig = go.Figure() # Create a new figure for plotting

fig.add_trace(go.Scatter(  # Add the actual values of the target variable to the plot
    x=y_test.index, 
    y=y_test, 
    mode='lines', 
    name='Actual T',
    line=dict(color='black')  
))   

fig.add_trace(go.Scatter( # Add the SARIMAX forecasted values to the plot
    x=y_test.index,
    y=sarimax_forecast,
    mode='lines',
    name='SARIMAX Forecast',
    line=dict(color='blue')
))

fig.update_layout(             # Update the layout of the plot
    title=f'SARIMAX Model Forecast vs Actual - {site_df}', 
    xaxis_title='Date', 
    yaxis_title='Cement Consumed Tonnes',
    legend=dict(x=0, y=1),
    template='plotly_white',
    width=1000, 
    height=450
)

fig.show()

In [22]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)  # Create a Random Forest Regressor model

In [23]:
rf.fit(X_train, y_train)  # Fit the Random Forest model to the training data

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [24]:
rf_forecast = rf.predict(X_test)  # Generate forecasts for the test period using the fitted Random Forest model

In [25]:
fig.add_trace(go.Scatter(  # Add the actual values of the target variable to the plot
    x=y_test.index, 
    y=y_test, 
    mode='lines', 
    name='Actual T',
    line=dict(color='black')  
))   

fig.add_trace(go.Scatter( # Add the Random Forest forecasted values to the plot
    x=y_test.index,
    y=rf_forecast,
    mode='lines',
    name='Random Forest Forecast',
    line=dict(color='red')
))

fig.update_layout(             # Update the layout of the plot
    title=f'Random Forest Model Forecast vs Actual - {site_df}', 
    xaxis_title='Date', 
    yaxis_title='Cement Consumed Tonnes',
    legend=dict(x=0, y=1),
    template='plotly_white',
    width=1000, 
    height=450
)

fig.show()

In [26]:
import numpy as np
from sklearn.metrics import mean_squared_error

def print_metrics(y_true, y_pred, label):
    mask = y_true != 0  # Create a mask to filter out zero values in the true values
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100      # Calculate Mean Absolute Percentage Error (MAPE)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))         # Calculate Root Mean Squared Error (RMSE)
    print(f"{label} - MAPE: {mape:.2f}%, RMSE: {rmse:.2f} tonnes")                     # Print the calculated metrics with the provided label

In [27]:
print_metrics(y_test, sarimax_forecast, "SARIMAX")  # Print metrics for the SARIMAX model
print_metrics(y_test, rf_forecast, "Random Forest")  # Print metrics for the Random Forest model

SARIMAX - MAPE: 36.17%, RMSE: 11.73 tonnes
Random Forest - MAPE: 34.86%, RMSE: 10.59 tonnes
